# 🇩🇪 RAGScore — Deutsche QA-Generierung & RAG-Bewertung

RAGScore erkennt deutsche Dokumente automatisch und generiert QA-Paare auf Deutsch.

| Funktion | Beschreibung |
|----------|-------------|
| 🌍 Automatische Spracherkennung | Erkennt deutsche Wörter und Umlaute (ä, ö, ü, ß) |
| 🎯 `--audience` | Zielgruppenspezifische Fragengenerierung |
| 📋 `--purpose` | Zweckbezogene Fragengenerierung |

**Voraussetzungen:** `ragscore >= 0.7.5`, OpenAI API-Schlüssel (oder anderer unterstützter Anbieter)

## 1. Installation

In [ ]:
!pip install -q "ragscore[notebook]" openai numpy

import nest_asyncio
nest_asyncio.apply()
print("✅ Bereit")

## 2. API-Schlüssel

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✅ API-Schlüssel aus Colab Secrets geladen")
except Exception:
    os.environ["OPENAI_API_KEY"] = "sk-..."  # Schlüssel hier eingeben
    print("⚠️  Hartcodierter Schlüssel — Colab Secrets empfohlen")

## 3. Deutsches Beispieldokument erstellen

In [ ]:
%%writefile deutsches_dokument.txt
Die Energiewende ist eine der größten Herausforderungen und Chancen für Deutschland im 21. Jahrhundert. Sie bezeichnet den Übergang von fossilen Energieträgern und Kernenergie hin zu erneuerbaren Energien wie Wind-, Solar- und Wasserkraft. Das Ziel der Bundesregierung ist es, bis 2045 klimaneutral zu werden und den Anteil erneuerbarer Energien am Bruttostromverbrauch auf mindestens 80 Prozent bis 2030 zu steigern.

Die Windenergie ist der wichtigste Pfeiler der deutschen Energiewende. Ende 2023 waren in Deutschland rund 30.000 Onshore-Windenergieanlagen mit einer installierten Leistung von etwa 60 Gigawatt in Betrieb. Die Offshore-Windenergie in Nord- und Ostsee trägt zusätzlich etwa 8 Gigawatt bei. Das Erneuerbare-Energien-Gesetz (EEG) bildet die gesetzliche Grundlage für den Ausbau und regelt die Einspeisevergütung für Strom aus erneuerbaren Quellen.

Die Solarenergie hat in den letzten Jahren einen enormen Zuwachs erlebt. Deutschland verfügt über eine installierte Photovoltaikleistung von über 80 Gigawatt. Die Kosten für Solarmodule sind seit 2010 um mehr als 90 Prozent gesunken, was die Wirtschaftlichkeit erheblich verbessert hat. Balkonkraftwerke mit einer Leistung von bis zu 800 Watt erfreuen sich bei Privathaushalten großer Beliebtheit und können ohne aufwendige Genehmigungsverfahren installiert werden.

Eine zentrale Herausforderung der Energiewende ist die Speicherung von Energie. Da Wind und Sonne nicht konstant verfügbar sind, werden Batteriespeicher, Pumpspeicherkraftwerke und Wasserstofftechnologien benötigt. Die Bundesregierung hat eine Nationale Wasserstoffstrategie verabschiedet, die Investitionen von 9 Milliarden Euro vorsieht. Grüner Wasserstoff, der durch Elektrolyse mit erneuerbarem Strom erzeugt wird, soll insbesondere in der Industrie und im Schwerlastverkehr eingesetzt werden.

Der Netzausbau ist ein weiterer kritischer Faktor. Neue Hochspannungsleitungen wie SuedLink (700 km) und SuedOstLink (540 km) sollen Windstrom von Nord- nach Süddeutschland transportieren. Die geschätzten Kosten für den gesamten Netzausbau belaufen sich auf über 30 Milliarden Euro. Gleichzeitig werden intelligente Stromnetze (Smart Grids) entwickelt, die Angebot und Nachfrage in Echtzeit ausbalancieren können.

Die wirtschaftlichen Auswirkungen der Energiewende sind erheblich. Der Sektor der erneuerbaren Energien beschäftigt in Deutschland über 300.000 Menschen. Die Strompreise für Verbraucher gehören jedoch zu den höchsten in Europa, was teilweise auf die EEG-Umlage und Netzentgelte zurückzuführen ist. Die Industrie befürchtet Wettbewerbsnachteile, weshalb energieintensive Unternehmen teilweise von der EEG-Umlage befreit werden. Die Gesamtinvestitionen in die Energiewende werden bis 2045 auf über 500 Milliarden Euro geschätzt.

## 4. Automatische Spracherkennung testen

In [ ]:
from ragscore.llm import detect_language

with open("deutsches_dokument.txt") as f:
    text = f.read()

lang = detect_language(text)
print(f"Erkannte Sprache: {lang}")  # Sollte 'de' anzeigen
assert lang == "de", f"Erwartet: 'de', Erkannt: '{lang}'"
print("✅ Deutsch wurde korrekt erkannt")

## 5. Mini-RAG erstellen

In [ ]:
import numpy as np
from openai import OpenAI

client = OpenAI()

with open("deutsches_dokument.txt") as f:
    text = f.read()
chunks = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 50]
print(f"✅ {len(chunks)} Chunks erstellt")

def get_embedding(text):
    return client.embeddings.create(input=text, model="text-embedding-3-small").data[0].embedding

chunk_embeddings = np.array([get_embedding(c) for c in chunks])
print(f"✅ {len(chunk_embeddings)} Chunks eingebettet")

def my_rag(question):
    """Frage einbetten → Top-3-Chunks finden → GPT-4o fragen"""
    q_emb = np.array(get_embedding(question))
    sims = chunk_embeddings @ q_emb / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(q_emb)
    )
    top_idx = np.argsort(sims)[-3:][::-1]
    context = "\n\n".join([chunks[i] for i in top_idx])

    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Antworten Sie nur basierend auf dem bereitgestellten Kontext. Seien Sie präzise und genau."},
            {"role": "user", "content": f"Kontext:\n{context}\n\nFrage: {question}"},
        ],
        temperature=0.3,
    )
    return resp.choices[0].message.content

print("\n🧪 Funktionstest:")
print(my_rag("Was ist die Energiewende?"))

## 6. Deutsche RAG-Bewertung (Baseline)

In [ ]:
from ragscore import quick_test

result = quick_test(my_rag, docs="deutsches_dokument.txt", n=5)
print("\n📋 Generierte Fragen:")
for _, row in result.df.iterrows():
    print(f"  F: {row['question'][:80]}")

## 7. 🎯 Zielgruppe: Ingenieure

In [ ]:
engineer_result = quick_test(
    my_rag, docs="deutsches_dokument.txt", n=5,
    audience="Ingenieure und Techniker",
    purpose="technische Planung",
)
print("\n⚙️ Ingenieur-Fragen:")
for _, row in engineer_result.df.iterrows():
    print(f"  F: {row['question'][:80]}")

## 8. 🎯 Zielgruppe: Politische Entscheidungsträger

In [ ]:
policy_result = quick_test(
    my_rag, docs="deutsches_dokument.txt", n=5,
    audience="politische Entscheidungsträger und Abgeordnete",
    purpose="Gesetzgebung und Regulierung",
)
print("\n🏛️ Fragen für Entscheidungsträger:")
for _, row in policy_result.df.iterrows():
    print(f"  F: {row['question'][:80]}")

## 9. Ergebnisse vergleichen

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Zielgruppe": ["Baseline", "Ingenieure", "Entscheidungsträger"],
    "Genauigkeit": [
        f"{result.accuracy:.0%}",
        f"{engineer_result.accuracy:.0%}",
        f"{policy_result.accuracy:.0%}",
    ],
    "Durchschn. Bewertung": [
        f"{result.avg_score:.1f}/5",
        f"{engineer_result.avg_score:.1f}/5",
        f"{policy_result.avg_score:.1f}/5",
    ],
    "Beispielfrage": [
        result.df.iloc[0]["question"][:60] + "..." if len(result.df) > 0 else "N/A",
        engineer_result.df.iloc[0]["question"][:60] + "..." if len(engineer_result.df) > 0 else "N/A",
        policy_result.df.iloc[0]["question"][:60] + "..." if len(policy_result.df) > 0 else "N/A",
    ],
})
display(comparison)

---

## 📚 Ressourcen

- **GitHub**: https://github.com/HZYAI/RagScore
- **PyPI**: https://pypi.org/project/ragscore/
- **Website**: https://ragscore.io

⭐ Wenn RAGScore Ihnen hilft, geben Sie uns einen Stern auf GitHub!